In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.ustable_cp import UStableConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs",
    loss_name=loss_name,
    loss_params=loss_params,
)

Instantiate region predictor

In [7]:
conformal_predictor = UStableConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
prediction_regions

[{'upper': [array([-1.70282203]),array([1.70872702])],
  'lower': [array([-1.68154543]),array([1.68745042])]},
 {'upper': [array([-1.69922359]),array([1.7123307])],
  'lower': [array([-1.677947]),array([1.6910541])]},
 {'upper': [array([-1.70131108]),array([1.71024107])],
  'lower': [array([-1.68003448]),array([1.68896448])]},
 {'upper': [array([-1.69903179]),array([1.71252371])],
  'lower': [array([-1.6777552]),array([1.69124711])]},
 {'upper': [array([-1.70367325]),array([1.70787533])],
  'lower': [array([-1.68239665]),array([1.68659873])]},
 {'upper': [array([-1.70174623]),array([1.70980263])],
  'lower': [array([-1.68046963]),array([1.68852603])]},
 {'upper': [array([-1.70140484]),array([1.71014656])],
  'lower': [array([-1.68012825]),array([1.68886996])]},
 {'upper': [array([-1.69982093]),array([1.71173156])],
  'lower': [array([-1.67854433]),array([1.69045497])]},
 {'upper': [array([-1.70123477]),array([1.71031798])],
  'lower': [array([-1.67995817]),array([1.68904138])]},
 {'upp

In [10]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.904


In [11]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.896
